# Baseline Evaluation - Production-Adapted News Summary

This notebook measures the baseline quality of one model with the current production-adapted prompt structure on `sunnysai12345/news-summary`.
Use the generated artifacts as the frozen baseline before creating a separate fine-tuning notebook.


In [ ]:
from pathlib import Path
import sys

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    print(f'Running outside Colab: {exc}')

DEFAULT_DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path.cwd()
SEARCH_ROOTS = [Path.cwd(), Path.cwd().parent, Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()]
REPO_ROOT = None
for candidate in SEARCH_ROOTS:
    if (candidate / 'reasoning_nlp').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT =', REPO_ROOT)
print('DEFAULT_DRIVE_ROOT =', DEFAULT_DRIVE_ROOT)


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'kaggle': 'kaggle',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'rouge_score': 'rouge-score',
    'transformers': 'transformers',
    'torch': 'torch',
    'accelerate': 'accelerate',
    'sentencepiece': 'sentencepiece',
    'bert_score': 'bert-score',
}
missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Python packages already satisfied')


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from reasoning_nlp.eval.news_summary_baseline import (
    NewsSummaryBaselineConfig,
    build_production_adapted_prompt_profile,
    run_baseline_evaluation,
)

DATASET_SLUG = os.environ.get('KAGGLE_DATASET_SLUG', 'sunnysai12345/news-summary').strip()
CACHE_DIR = Path(os.environ.get('NEWS_SUMMARY_CACHE_DIR', str(DEFAULT_DRIVE_ROOT / 'video-summary-cache' / 'news-summary')))
RESULTS_DIR = Path(os.environ.get('NEWS_SUMMARY_RESULTS_DIR', str(DEFAULT_DRIVE_ROOT / 'video-summary-eval')))
KAGGLE_JSON_DRIVE_PATH = Path(os.environ.get('KAGGLE_JSON_DRIVE_PATH', str(DEFAULT_DRIVE_ROOT / '.kaggle' / 'kaggle.json')))

CSV_FILENAME = 'news_summary.csv'
ARTICLE_COLUMN = 'ctext'
SUMMARY_COLUMN = 'text'
AUX_HEADLINE_COLUMN = 'headlines'
SPLIT_COLUMN = os.environ.get('NEWS_SUMMARY_SPLIT_COLUMN', '').strip()
TARGET_SPLIT = os.environ.get('NEWS_SUMMARY_TARGET_SPLIT', 'test').strip()
USE_FIXED_SPLIT = os.environ.get('NEWS_SUMMARY_USE_FIXED_SPLIT', '1').strip().lower() not in {'0', 'false', 'no'}

MODEL_NAME = os.environ.get('VIDEO_SUMMARY_LOCAL_MODEL_VERSION', 'Qwen/Qwen2.5-3B-Instruct').strip()
BACKEND = os.environ.get('VIDEO_SUMMARY_EVAL_BACKEND', 'local').strip().lower()
OPENAI_MODEL = os.environ.get('OPENAI_MODEL', '').strip()

EVAL_PROTOCOL_VERSION = os.environ.get('NEWS_SUMMARY_EVAL_PROTOCOL_VERSION', 'news-summary-baseline-v1').strip()
MAX_SAMPLES = int(os.environ.get('NEWS_SUMMARY_MAX_SAMPLES', '128'))
RANDOM_SEED = int(os.environ.get('NEWS_SUMMARY_RANDOM_SEED', '42'))
BATCH_SIZE = int(os.environ.get('NEWS_SUMMARY_BATCH_SIZE', '4'))
MAX_INPUT_CHARS = int(os.environ.get('NEWS_SUMMARY_MAX_INPUT_CHARS', '6000'))
MAX_INPUT_TOKENS = int(os.environ.get('NEWS_SUMMARY_MAX_INPUT_TOKENS', '3072'))
MAX_NEW_TOKENS = int(os.environ.get('NEWS_SUMMARY_MAX_NEW_TOKENS', '96'))
SPOTCHECK_SAMPLE_SIZE = int(os.environ.get('NEWS_SUMMARY_SPOTCHECK_SAMPLE_SIZE', '24'))
ENABLE_BERTSCORE = os.environ.get('NEWS_SUMMARY_ENABLE_BERTSCORE', '1').strip().lower() not in {'0', 'false', 'no'}
SAVE_PREDICTIONS_WITH_ARTICLE = os.environ.get('NEWS_SUMMARY_SAVE_PREDICTIONS_WITH_ARTICLE', '0').strip().lower() in {'1', 'true', 'yes'}
FORCE_REDOWNLOAD = os.environ.get('NEWS_SUMMARY_FORCE_REDOWNLOAD', '0').strip().lower() in {'1', 'true', 'yes'}
SAFE_MODEL_NAME = MODEL_NAME.replace('/', '_')

FROZEN_EVAL_IDS_PATH = Path(
    os.environ.get(
        'NEWS_SUMMARY_FROZEN_IDS_PATH',
        str(RESULTS_DIR / 'protocol' / f"{EVAL_PROTOCOL_VERSION}_{SAFE_MODEL_NAME}_frozen_eval_ids.csv"),
    )
)

config = NewsSummaryBaselineConfig(
    protocol_version=EVAL_PROTOCOL_VERSION,
    dataset_slug=DATASET_SLUG,
    cache_dir=CACHE_DIR,
    results_dir=RESULTS_DIR,
    kaggle_json_drive_path=KAGGLE_JSON_DRIVE_PATH,
    csv_filename=CSV_FILENAME,
    article_column=ARTICLE_COLUMN,
    summary_column=SUMMARY_COLUMN,
    aux_headline_column=AUX_HEADLINE_COLUMN,
    split_column=SPLIT_COLUMN,
    target_split=TARGET_SPLIT,
    use_fixed_split=USE_FIXED_SPLIT,
    frozen_eval_ids_path=FROZEN_EVAL_IDS_PATH,
    model_name=MODEL_NAME,
    backend=BACKEND,
    openai_model=OPENAI_MODEL,
    max_samples=MAX_SAMPLES,
    random_seed=RANDOM_SEED,
    batch_size=BATCH_SIZE,
    max_input_chars=MAX_INPUT_CHARS,
    max_input_tokens=MAX_INPUT_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
    enable_bertscore=ENABLE_BERTSCORE,
    save_predictions_with_article=SAVE_PREDICTIONS_WITH_ARTICLE,
    spotcheck_sample_size=SPOTCHECK_SAMPLE_SIZE,
)
prompt_profile = build_production_adapted_prompt_profile()

print('MODEL_NAME =', MODEL_NAME)
print('BACKEND =', BACKEND)
print('DATASET_SLUG =', DATASET_SLUG)
print('FROZEN_EVAL_IDS_PATH =', FROZEN_EVAL_IDS_PATH)
display(Markdown(f'**Prompt profile:** `{prompt_profile.name}` | `{prompt_profile.prompt_version}`'))
display(Markdown(prompt_profile.adaptation_note))


In [ ]:
result = run_baseline_evaluation(
    config=config,
    prompt_profile=prompt_profile,
    force_redownload=FORCE_REDOWNLOAD,
)
run_dir = result['run_dir']
metrics = result['metrics']
comparison_df = result['comparison_df']
per_example_df = result['per_example_df']
error_analysis_df = result['error_analysis_df']
spotcheck_df = result['spotcheck_df']

print('Artifacts saved to', run_dir)
display(pd.DataFrame([result['dataset_profile']]))
display(comparison_df)
display(error_analysis_df.head(10))
display(spotcheck_df.head(10))


In [ ]:
key_metrics = pd.DataFrame(
    [
        {'metric': 'rouge1', 'value': metrics.get('rouge1')},
        {'metric': 'rouge2', 'value': metrics.get('rouge2')},
        {'metric': 'rougeL', 'value': metrics.get('rougeL')},
        {'metric': 'bertscore_f1', 'value': metrics.get('bertscore_f1')},
        {'metric': 'parse_success_rate', 'value': metrics.get('parse_success_rate')},
        {'metric': 'hallucination_proxy_rate', 'value': metrics.get('hallucination_proxy_rate')},
        {'metric': 'instruction_leakage_rate', 'value': metrics.get('instruction_leakage_rate')},
    ]
)
display(key_metrics)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(key_metrics['metric'], key_metrics['value'].fillna(0.0))
axes[0].set_title('Key automatic metrics')
axes[0].tick_params(axis='x', rotation=45)

per_example_ok = per_example_df.loc[per_example_df['status'].eq('ok')].copy()
if not per_example_ok.empty:
    axes[1].scatter(per_example_ok['latency_ms'], per_example_ok['rougeL'], alpha=0.6)
    axes[1].set_title('Latency vs ROUGE-L')
    axes[1].set_xlabel('latency_ms')
    axes[1].set_ylabel('rougeL')
else:
    axes[1].set_title('No scored examples')
plt.tight_layout()
plt.show()


In [ ]:
report_path = run_dir / 'baseline_report.md'
display(Markdown(report_path.read_text(encoding='utf-8')))


In [ ]:
HUMAN_EVAL_PATH = Path(os.environ.get('NEWS_SUMMARY_HUMAN_EVAL_PATH', '')).expanduser() if os.environ.get('NEWS_SUMMARY_HUMAN_EVAL_PATH', '').strip() else None
if HUMAN_EVAL_PATH and HUMAN_EVAL_PATH.exists():
    human_df = pd.read_csv(HUMAN_EVAL_PATH)
    summary = {
        'faithfulness_mean': float(pd.to_numeric(human_df['faithfulness_score'], errors='coerce').mean()),
        'coverage_mean': float(pd.to_numeric(human_df['coverage_score'], errors='coerce').mean()),
        'fluency_mean': float(pd.to_numeric(human_df['fluency_score'], errors='coerce').mean()),
        'hallucination_rate': float(human_df['hallucination_flag'].astype(str).str.strip().str.lower().isin({'1', 'true', 'yes', 'y'}).mean()),
        'usable_for_training_target_rate': float(human_df['usable_for_training_target'].astype(str).str.strip().str.lower().isin({'1', 'true', 'yes', 'y'}).mean()),
    }
    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(run_dir / 'human_eval_summary.csv', index=False)
    display(summary_df)
else:
    print('Optional: set NEWS_SUMMARY_HUMAN_EVAL_PATH to a filled spotcheck CSV to aggregate human evaluation.')
